# CARE to Compare: reproducing the paper results

This notebook reproduces the results of the CARE paper ([doi:10.3390/data9120138](https://doi.org/10.3390/data9120138))
on the CARE to Compare dataset ([zenodo](https://doi.org/10.5281/zenodo.10958774)) with the published model configs
in `c2c_configs/`. For an introduction to `Care2CompareDataset` and `CAREScore`, see `CARE to Compare.ipynb`.

Training an autoencoder is random (initial weights, validation split, noise), so a single model per event does not
give exactly the paper results. As advised in the paper, every event is therefore trained **10 times**, and the best
of the 10 runs per event is kept:

1. Train a model for each of the 95 events, 10 times (950 models).
2. Evaluate every run with the CARE score.
3. Keep the best run per event and calculate the final CARE score.

The results of all 10 runs are saved in `results/`. Running this notebook again skips every event that already has a
result, so with the saved results nothing is trained. To train everything again, set `results_dir` to a new folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from energy_fault_detector.fault_detector import FaultDetector
from energy_fault_detector.config import Config
from energy_fault_detector.evaluation import CAREScore, Care2CompareDataset

2026-11-12 17:35:49.801105: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-12 17:35:49.801162: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-12 17:35:49.802745: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-12 17:35:49.817339: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-12 17:35:50.936733: W tensorflow/compiler/tf2

## Settings

In [2]:
data_dir = Path('..') / '..' / 'CARE_To_Compare'
results_dir = Path('results')
n_runs = 10
wind_farms = ['A', 'B', 'C']

In [3]:
c2c = Care2CompareDataset(path=data_dir, download_dataset=False)  # set download_dataset=True to download it first

configs = {wf: Config(f'c2c_configs/windfarm_{wf}.yaml') for wf in wind_farms}
for wf in wind_farms:
    c2c.update_c2c_config(configs[wf], wf)

# WF A: the normal index is not based on SCADA status codes, so it is ignored in the evaluation
ignore_normal_idx = {'A': True, 'B': False, 'C': False}

**Wind farm C training filter.** As in the paper, rows with a wind speed outside the normal operation range (below
cut-in or above cut-off) or with (almost) no power are treated as anomalous status in the training data, so the model
does not learn them as normal behaviour.

In [4]:
wind_col, power_col = 'Wind speed 1 (avg)', 'Active power HV grid (avg)'
cut_in, cut_off, min_power = 3, 25, 0.01  # m/s, m/s, kW

## 1. Train and evaluate 10 runs per event

Each model is trained on the training period of its event and evaluated on the prediction period, as in
`CARE to Compare.ipynb`. Every model gets a new random seed, which is saved in the results. `id` is only a row
counter, so it is dropped together with the other columns that are not sensor data.

The results are saved after every event as `results/run_<run>/results_<wind farm>.csv`.

In [5]:
for run in range(1, n_runs + 1):
    for wf in wind_farms:
        results_file = results_dir / f'run_{run}' / f'results_{wf}.csv'
        results_file.parent.mkdir(parents=True, exist_ok=True)
        care_score = CAREScore(coverage_beta=0.5, reliability_beta=0.5, anomaly_detection_method='criticality')
        if results_file.exists():
            care_score.load_evaluated_events(results_file)

        for event_id in c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == wf, 'event_id']:
            if not care_score.evaluated_events.empty and event_id in care_score.evaluated_events['event_id'].values:
                continue  # skip if already evaluated
            print('Run', run, 'wind farm', wf, 'event', event_id)

            x_train, x_test = c2c.load_event_dataset(event_id=event_id, index_column='time_stamp')
            # normal index
            y_train = x_train['status_type_id'] == 0
            y_test = x_test['status_type_id'] == 0
            if wf == 'C':
                in_operation = x_train[wind_col].between(cut_in, cut_off) & (x_train[power_col] >= min_power)
                y_train = np.logical_and(y_train, in_operation)
            # drop columns that are not sensor data
            x_train = x_train.drop(['id', 'asset_id', 'status_type_id'], axis=1, errors='ignore')
            x_test = x_test.drop(['id', 'asset_id', 'status_type_id'], axis=1, errors='ignore')

            # train and predict
            seed = int(np.random.default_rng().integers(2**31 - 1))
            tf.keras.utils.set_random_seed(seed)
            model = FaultDetector(configs[wf])
            model.fit(sensor_data=x_train, normal_index=y_train, save_models=False)
            prediction = model.predict(x_test)

            # evaluate event
            event_info = c2c.event_info_all[c2c.event_info_all['event_id'] == event_id]
            evaluation = care_score.evaluate_event(
                event_id=event_id,
                event_start=event_info['event_start'].iloc[0],
                event_end=event_info['event_end'].iloc[0],
                event_label=event_info['event_label'].iloc[0],
                normal_index=y_test,
                predicted_anomalies=prediction.predicted_anomalies,
                ignore_normal_index=ignore_normal_idx[wf],
            )
            evaluation['seed'] = seed
            care_score.save_evaluated_events(results_file)

## 2. CARE score of every run

The CARE score of each run per wind farm and over all 95 events, with its four parts. A fault counts as detected when
the criticality reaches 72 before the end of the event; a normal event with such an alarm is a false alarm. The summary
of each run is saved as `results/run_<run>/summary.csv`.

In [6]:
summaries = {}
for run in range(1, n_runs + 1):
    run_dir = results_dir / f'run_{run}'
    all_evaluations = pd.concat([pd.read_csv(run_dir / f'results_{wf}.csv') for wf in wind_farms])
    all_evaluations.to_csv(run_dir / 'results_all.csv', index=False)
    care_score = CAREScore(coverage_beta=0.5, reliability_beta=0.5, anomaly_detection_method='criticality')
    care_score.load_evaluated_events(run_dir / 'results_all.csv')

    summary = {}
    for name in wind_farms + ['all']:
        events = c2c.event_info_all['event_id'] if name == 'all' else \
            c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == name, 'event_id']
        selected = all_evaluations[all_evaluations['event_id'].isin(events)]
        summary[name] = {
            'CARE': care_score.get_final_score(event_selection=events),
            'coverage': care_score.calculate_avg_coverage(event_selection=events),
            'accuracy': care_score.calculate_avg_accuracy(event_selection=events),
            'reliability': care_score.calculate_reliability(event_selection=events),
            'earliness': care_score.calculate_avg_earliness(event_selection=events),
            'detected faults': selected.loc[selected['event_label'] == 'anomaly', 'anomaly_detected'].sum(),
            'false alarms': selected.loc[selected['event_label'] == 'normal', 'anomaly_detected'].sum(),
        }
    summaries[run] = pd.DataFrame(summary).T.astype({'detected faults': int, 'false alarms': int})
    summaries[run].to_csv(run_dir / 'summary.csv', index_label='wind_farm')

CARE score of each run:

In [7]:
care_per_run = pd.DataFrame({f'run {run}': summaries[run]['CARE'] for run in summaries}).T
care_per_run.loc['mean'] = care_per_run.mean()
care_per_run.loc['std'] = care_per_run.iloc[:n_runs].std()
care_per_run.astype(float).round(3)

,A,B,C,all
run 1,0.593,0.607,0.639,0.617
run 2,0.563,0.638,0.643,0.619
run 3,0.614,0.594,0.623,0.608
run 4,0.615,0.591,0.627,0.609
run 5,0.613,0.582,0.636,0.613
run 6,0.565,0.644,0.635,0.616
run 7,0.612,0.570,0.643,0.615
run 8,0.583,0.668,0.647,0.630
run 9,0.611,0.604,0.637,0.616
run 10,0.616,0.634,0.641,0.626


## 3. Best run per event

All 950 evaluated events (95 events × 10 runs):

In [8]:
all_runs = pd.concat([pd.read_csv(results_dir / f'run_{run}' / 'results_all.csv').assign(run=run)
                      for run in range(1, n_runs + 1)], ignore_index=True)
all_runs

,event_id,event_label,weighted_score,max_criticality,first_time_detected,early_detection_time,f_beta_score,accuracy,tn,fp,fn,tp,seed,anomaly_detected,run
0,68,anomaly,0.114249,12,NaN,NaN,0.352697,0.181699,179,102,1776,238,1409724761,False,1
1,22,anomaly,0.055449,36,NaN,NaN,0.352113,0.188153,101,42,890,115,1461337755,False,1
2,72,anomaly,0.110181,10,NaN,NaN,0.250878,0.476946,862,146,909,100,93669246,False,1
3,73,anomaly,0.138545,26,NaN,NaN,0.410362,0.454020,606,114,830,179,1772530477,False,1
4,0,anomaly,0.161146,30,NaN,NaN,0.284402,0.222692,341,485,1721,291,1931778954,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,37,normal,NaN,620,NaN,NaN,NaN,0.495663,1200,1221,0,0,120736958,True,10
946,29,normal,NaN,60,NaN,NaN,NaN,0.901840,2058,224,0,0,1481541974,False,10
947,1,normal,NaN,12,NaN,NaN,NaN,0.993399,2107,14,0,0,1040726806,False,10
948,20,normal,NaN,53,NaN,NaN,NaN,0.937400,2336,156,0,0,352912528,False,10


From these, the best run of each event is kept: for an anomaly event the run with the highest coverage + earliness
(the fault period is flagged most and earliest), for a normal event the run with the highest accuracy (fewest normal
rows flagged). On a tie the earliest run is taken. The selection is saved as `results/best_per_event.csv`.

Note: the best run is selected on the prediction period itself, so this score shows what the configs can reach, not
what one trained model reaches (see the mean of the single runs above).

In [9]:
all_runs['quality'] = np.where(all_runs['event_label'] == 'anomaly',
                               all_runs['f_beta_score'].fillna(0) + all_runs['weighted_score'].fillna(0),
                               all_runs['accuracy'].fillna(0))
best = (all_runs.sort_values(['event_id', 'quality', 'run'], ascending=[True, False, True])
        .drop_duplicates('event_id')
        .drop(columns='quality'))
best.to_csv(results_dir / 'best_per_event.csv', index=False)
best

,event_id,event_label,weighted_score,max_criticality,first_time_detected,early_detection_time,f_beta_score,accuracy,tn,fp,fn,tp,seed,anomaly_detected,run
859,0,anomaly,0.160805,32,NaN,NaN,0.287855,0.227273,353,473,1720,292,2987212,False,10
757,1,normal,NaN,12,NaN,NaN,NaN,0.993871,2108,13,0,0,1195353932,False,8
696,2,normal,NaN,71,NaN,NaN,NaN,0.734259,1586,574,0,0,1912627168,False,8
490,3,normal,NaN,37,NaN,NaN,NaN,0.903695,2984,318,0,0,1542377800,False,6
801,4,anomaly,0.382213,322,2023-08-01 04:20:00,18 days 05:40:00,0.723632,0.518833,925,11,1688,907,736630747,True,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,90,anomaly,0.659740,1000,2023-08-09 14:30:00,13 days 03:00:00,0.881880,0.761634,377,91,370,1096,1472411869,True,1
627,91,anomaly,0.111696,111,2023-10-18 13:30:00,13 days 22:30:00,0.237681,0.328539,1001,83,2298,164,1356991736,True,7
20,92,normal,NaN,8,NaN,NaN,NaN,0.948703,1683,91,0,0,1233510228,False,1
173,93,normal,NaN,81,NaN,NaN,NaN,0.893111,2891,346,0,0,980911689,True,2


## Final CARE score

CARE score of the best run per event, per wind farm and over all 95 events.

In [10]:
care_score = CAREScore(coverage_beta=0.5, reliability_beta=0.5, anomaly_detection_method='criticality')
care_score.load_evaluated_events(results_dir / 'best_per_event.csv')

final = {}
for name in wind_farms + ['all']:
    events = c2c.event_info_all['event_id'] if name == 'all' else \
        c2c.event_info_all.loc[c2c.event_info_all['wind_farm'] == name, 'event_id']
    selected = best[best['event_id'].isin(events)]
    final[name] = {
        'CARE': round(care_score.get_final_score(event_selection=events), 3),
        'coverage': round(care_score.calculate_avg_coverage(event_selection=events), 3),
        'accuracy': round(care_score.calculate_avg_accuracy(event_selection=events), 3),
        'reliability': round(care_score.calculate_reliability(event_selection=events), 3),
        'earliness': round(care_score.calculate_avg_earliness(event_selection=events), 3),
        'detected faults': f"{selected.loc[selected['event_label'] == 'anomaly', 'anomaly_detected'].sum()} / {(selected['event_label'] == 'anomaly').sum()}",
        'false alarms': f"{selected.loc[selected['event_label'] == 'normal', 'anomaly_detected'].sum()} / {(selected['event_label'] == 'normal').sum()}",
    }
pd.DataFrame(final).T

,CARE,coverage,accuracy,reliability,earliness,detected faults,false alarms
A,0.619,0.48,0.898,0.625,0.198,3 / 12,0 / 10
B,0.713,0.76,0.778,0.714,0.534,6 / 6,3 / 9
C,0.674,0.528,0.868,0.599,0.506,20 / 27,15 / 31
all,0.662,0.546,0.858,0.622,0.428,29 / 45,18 / 50


## Details per event

The best run of every event, per wind farm. `days early` is the time between the first alarm (criticality ≥ 72) and
the end of the event; `run` and `seed` show which run was selected.

In [11]:
details = best.merge(c2c.event_info_all[['event_id', 'wind_farm']], on='event_id').set_index(['wind_farm', 'event_id']).sort_index()
details['days early'] = pd.to_timedelta(details['early_detection_time']).dt.total_seconds() / (24 * 60 * 60)

**Anomaly events:** was the fault detected (`detected`), how early, and how much of the event period was flagged
(`coverage`).

In [14]:
details.loc[details['event_label'] == 'anomaly',
            ['anomaly_detected', 'days early', 'f_beta_score', 'weighted_score', 'max_criticality', 'run']] \
    .rename(columns={'anomaly_detected': 'detected', 'f_beta_score': 'coverage', 'weighted_score': 'earliness'}).round(3)

detected  days early  coverage  earliness  \
wind_farm event_id                                              
A         0            False         NaN     0.288      0.161   
          10           False         NaN     0.506      0.237   
          22           False         NaN     0.395      0.076   
          26           False         NaN     0.398      0.115   
          40            True      18.139     0.739      0.287   
          42            True       0.257     0.545      0.169   
          45            True       6.028     0.672      0.429   
          51           False         NaN     0.380      0.118   
          68           False         NaN     0.420      0.165   
          72           False         NaN     0.329      0.154   
          73           False         NaN     0.540      0.307   
          84           False         NaN     0.543      0.152   
B         7             True      31.396     0.828      0.824   
          19            True      14.486     0.613      0.367   
          27            True      45.562     0.753      0.395   
          34            True       6.806     0.562      0.340   
          53            True      36.854     0.912      0.680   
          77            True      50.194     0.890      0.598   
C         4             True      18.236     0.724      0.382   
          5             True       1.597     0.806      0.605   
          9             True      16.799     0.409      0.163   
          11            True      20.854     1.000      1.000   
          12            True      15.868     0.580      0.314   
          15            True       9.278     0.312      0.105   
          16           False         NaN     0.355      0.912   
          18            True       5.403     0.058      0.036   
          28            True      16.750     0.861      0.761   
          30           False         NaN     0.514      0.829   
          31            True       0.903     0.695      0.897   
          33            True      20.410     0.939      1.000   
          35            True       2.090     0.699      0.191   
          39           False         NaN     0.127      0.541   
          44            True      64.389     0.990      0.949   
          47            True       4.271     0.506      0.971   
          49            True       0.403     0.345      0.071   
          55            True       7.771     0.391      0.150   
          66           False         NaN     0.000      0.713   
          67            True      46.250     0.304      0.114   
          70            True      19.194     0.868      0.884   
          76           False         NaN     0.648      0.504   
          78            True       0.403     0.587      0.228   
          79           False         NaN     0.000      0.187   
          81           False         NaN     0.424      0.390   
          90            True      13.125     0.882      0.660   
          91            True      13.938     0.238      0.112   

                    max_criticality  run  
wind_farm event_id                        
A         0                      32   10  
          10                     71    2  
          22                     38    9  
          26                     30    3  
          40                   1000    7  
          42                     76    3  
          45                    135   10  
          51                     31    3  
          68                     20    4  
          72                     14    6  
          73                     36   10  
          84                     51   10  
B         7                    1000    8  
          19                    112    8  
          27                    355    1  
          34                    505    9  
          53                   1000    2  
          77                   1000    9  
C         4                     322    9  
          5                     111    1  
          9       

**Normal events:** did the model raise a false alarm, and how many normal rows were flagged (`fp`) or not (`tn`).

In [15]:
details.loc[details['event_label'] == 'normal',
            ['anomaly_detected', 'max_criticality', 'accuracy', 'fp', 'tn', 'run']] \
    .rename(columns={'anomaly_detected': 'false alarm'}).round(3)

false alarm  max_criticality  accuracy    fp     tn  run
wind_farm event_id                                                          
A         3               False               37     0.904   318   2984    6
          13              False                8     0.912   314   3269    9
          14              False               14     0.860   267   1634   10
          17              False               29     0.877   342   2439   10
          24              False               37     0.909   248   2466    5
          25              False               13     0.927   176   2247    4
          38              False               17     0.869   333   2211    1
          69              False               18     0.916   219   2374    4
          71              False               35     0.855   356   2093    1
          92              False                8     0.949    91   1683    1
B         2               False               71     0.734   574   1586    8
          21               True              101     0.697   384    885    1
          23              False               34     0.741   451   1292   10
          52              False               42     0.895   286   2444   10
          74              False               56     0.783   654   2365    6
          82              False               47     0.765   562   1825    8
          83               True              153     0.820  2320  10539    9
          86              False               18     0.873   362   2494    7
          87               True              185     0.696   893   2041    5
C         1               False               12     0.994    13   2108    8
          6               False               50     0.941   124   1994    1
          8                True               79     0.917   183   2033    2
          20              False               52     0.943   143   2349    1
          29              False               61     0.906   215   2067    1
          32              False               33     0.970    72   2352    8
          36               True               83     0.868   376   2471    1
          37               True              620     0.496  1220   1201    2
          41               True              109     0.889   463   3727    7
          43              False               52     0.950   127   2420   10
          46               True              173     0.865   346   2208    8
          48              False               61     0.872   346   2355    4
          50              False               41     0.963   127   3283    3
          54              False               62     0.894   315   2643    5
          56               True              153     0.851   320   1822   10
          57              False               47     0.968    77   2325    9
          58              False               38     0.953    88   1769    1
          59               True              155     0.926   202   2514    5
          60              False               58     0.902   233   2143    1
          61               True              167     0.853   430   2489    8
          62              False               28     0.961    40    986    6
          63               True               86     0.769   528   1758    5
          64               True              210     0.846   284   1562    4
          65               True             1000     0.196  2668    652    1
          75               True              415     0.780   763   2713    1
          80               True              108     0.926   172   2156    1
          85              False               59     0.905   149   1412    9
          88              False                4     0.942   146   2385    1
          89              False               57     0.933   182   2526   10
          93               True               81     0.893   346   2891    2
          94               True               78     0.838   390   2020    2

# CARE Score usage on other datasets

To use the CARE-Score with other datasets you need the following data:
- define events containing anomalous data (the period before an actual fault)
- define events containing normal data
- For all events there needs to be enough training data beforehand to train an early fault detection model
- Ideally you have the same amount of normal and anomalous events

Then you can use the `CAREScore` class to calculate the final score:
- Create models and get predictions for each event.
- Evaluate each event using the `CAREScore.evaluate_event` method
- Calculate the CARE score `CAREScore.get_final_score`

For each of these events, you need to be able to train a proper model (for example one large model or a model for each event). For the CARE2Compare dataset we assumed 1 year of training data with >=70% normal operation is enough to create a normal behavior model.
